# Negative Log-Likelihood, Cross-Entropy, and Perplexity

負の対数尤度、交差エントロピー、パープレキシティの関係をまとめる。

In [1]:
import torch
import torch.nn.functional as F

torch.set_printoptions(sci_mode=False, precision=2)

## Negative Log-Likelihood (NLL)

文字通り、対数尤度にマイナスをつけたもの。

$$
\begin{align}
\text{NLL} = -\log p
\end{align}
$$

In [2]:
# モデルが出力するロジット
logits = torch.tensor([
    [3.2, -3.0, 0.0, 1.0],
    [0.6, 0.1, 0.0, 0.0],
    [-0.4, -2.0, 0.4, -3.0],
])

# 確率分布に変換
dists = F.softmax(logits, dim=-1)
dists

tensor([[0.87, 0.00, 0.04, 0.10],
        [0.37, 0.22, 0.20, 0.20],
        [0.29, 0.06, 0.64, 0.02]])

In [3]:
# 正解ラベル
labels = torch.tensor([0, 0, 2])

# 対応する尤度（確率）を取り出す
likelihoods = dists[torch.arange(len(labels)), labels]
likelihoods

tensor([0.87, 0.37, 0.64])

In [4]:
# 対数に変換
log_likelihoods = likelihoods.log()
log_likelihoods

tensor([-0.14, -0.99, -0.45])

In [5]:
# マイナスをつける
nll = -log_likelihoods
nll

tensor([0.14, 0.99, 0.45])

In [6]:
# 損失をとる場合は平均か和をとる
nll.mean()

tensor(0.53)

[torch.nn.functional.nll_loss](https://docs.pytorch.org/docs/2.12/generated/torch.nn.functional.nll_loss.html)に用意されてる。入力は確率の対数。

In [7]:
F.nll_loss(dists.log(), labels)

tensor(0.53)

モデルの出力に対して`log_softmax`を使うといい。

In [8]:
F.nll_loss(F.log_softmax(logits, dim=-1), labels)

tensor(0.53)

## Cross-Entropy

分類タスクにおける、予測分布と正解分布の差分。正解分布がone-hotベクトルである場合、NLLと同じになる。

$$
\begin{align}
\text{Cross-Entropy} = -\sum_{i=1}^d y_i \log p_i
\end{align}
$$

[torch.nn.functional.cross_entropy](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)に用意されてる。入力はロジットでよく、またone-hotベクトルを前提としているのでラベルのみを与えるだけでよい。

In [9]:
F.cross_entropy(logits, labels)

tensor(0.53)

## Perplexity

NLL（CE）の指数。予測確率の逆数と同値。「正解となるクラスを何択まで絞れているか」と解釈できるので、分岐数とも呼ばれる。

$$
\begin{align}
\text{Perplexity}
&= \exp(\text{NLL}) \\
&= \exp(-\log p) \\
&= \frac{1}{p}
\end{align}
$$

複数データの場合も同様にNLLの平均で求められ、この場合最終的な値は各Perplexityの幾何平均（相乗平均）となる。

In [10]:
ppl = torch.exp(nll.mean())
ppl

tensor(1.70)

In [11]:
# 幾何平均
def geometric_mean(x):
    return torch.pow(torch.prod(x), 1/len(x))

geometric_mean(1 / likelihoods)

tensor(1.70)